# 12 — Binomial revolve: recompute more, remember less

Notebook 11 closed Move 3 on a cliffhanger it named out loud: segmented fold
adjoints are *uniform* checkpointing (Chen's √T heuristic), **not** the optimal
**binomial revolve** of Griewank & Walther. This notebook is that refinement.

The thesis in one line: **recompute *more* than once, and the live memory drops
from √T toward log T — the schedule changes, the certified pieces do not.** The
reverse fold, the just-in-time trajectory recompute, the ŝ seam identity: every
lemma from notebook 08's fold adjoint stays exactly as it was. Revolve only
chooses *which* ranges to recompute and *in what order* — a recursive schedule
laid over the same store-all leaf. `grad(..., fold_slots=S)` is the whole new
surface: `S` checkpoint slots, and revolve does the rest.

In [1]:
import nbhelp  # noqa: F401  — puts tensorlib on sys.path
from math import comb

import numpy as np
from tensorlib import Tensor
from tensorlib.autodiff import _revolve_cost, _revolve_split, grad
from tensorlib.ir import Instr, Program, run
from tensorlib.memory import peak_memory
from tensorlib.opcount import ops_count


def I(var, op, operands=(), **params):
    return Instr(var, op, tuple(operands), params)


def T(arr, names):
    return Tensor.from_numpy(np.asarray(arr, dtype=np.float64), names)

In [2]:
# 1D FDTD leapfrog (notebook 08/11): state = (E on 6 nodes, H on 5 edges),
# no per-step elements. The step is a 17-instruction Program.
N, C = 6, 0.3
FDTD_STEP = Program(
    (
        I("E", "input"),
        I("H", "input"),
        I("Es", "shift", ["E"], deltas={"x": -1}),
        I("Ea", "slice", ["Es"], ranges={"x": (0, N - 1)}),
        I("Eb", "slice", ["E"], ranges={"x": (0, N - 1)}),
        I("dE", "pointwise", ["Ea", "Eb"], f="sub"),
        I("cH", "const", [], value=C, dims=(("x", (0, N - 1)),)),
        I("dHs", "pointwise", ["cH", "dE"], f="mul"),
        I("H1", "pointwise", ["H", "dHs"], f="add"),
        I("Hs", "shift", ["H1"], deltas={"x": 1}),
        I("Ha", "slice", ["H1"], ranges={"x": (1, N - 1)}),
        I("Hb", "slice", ["Hs"], ranges={"x": (1, N - 1)}),
        I("dH", "pointwise", ["Ha", "Hb"], f="sub"),
        I("pdH", "pad", ["dH"], fill=0.0, extents={"x": (0, N)}),
        I("cE", "const", [], value=C, dims=(("x", (0, N)),)),
        I("dEs", "pointwise", ["cE", "pdH"], f="mul"),
        I("E1", "pointwise", ["E", "dEs"], f="add"),
    )
)


def fdtd_prog(steps, out=("final", "E1")):
    dims = ("x",) if out[0] == "final" else ("t", "x")
    return Program(
        (
            I("E0", "input"),
            I("H0", "input"),
            I("Ef", "fold", ["E0", "H0"], step=FDTD_STEP, dim="t",
              state=("E", "H"), element=(), carry={"E": "E1", "H": "H1"},
              out=out, extent=(0, steps)),
            I("E2", "pointwise", ["Ef", "Ef"], f="mul"),
            I("loss", "reduce", ["E2"], f="sum", dims=dims),
        )
    )


E = np.zeros(N); E[2] = 1.0                          # a pulse
fin = {"E0": T(E, ("x",)), "H0": T(np.zeros(N - 1), ("x",))}
print("FDTD leapfrog; one (E, H) state =", (N + (N - 1)) * 8, "B")

FDTD leapfrog; one (E, H) state = 88 B


## Where notebook 11 left off — the uniform √T curve

A `fold` is a loop whose store-everything adjoint keeps **all T** step
trajectories alive to feed the reverse pass. `fold_segments=K` cuts the time
axis into K equal segments, keeps only the K **boundary** states up front, and
recomputes each segment's trajectory just-in-time — ~T/K + K states live instead
of T, minimized at K ≈ √T.

One change from notebook 11: the loss here is over the **final** field
(`out=("final","E1")`), not the whole trajectory. With `out=("emit",…)` the
entire space-time trajectory *is* the output — materialized regardless, which
masks any checkpointing win. `out=final` makes the trajectory a *pure backward
cost*, the honest setting for a memory experiment. Sweep K over the divisors of
T = 24:

In [3]:
prog = fdtd_prog(24)
print(f"{'K':>5}  {'segments':>13}  {'peak B':>7}  {'ops':>7}")
jp0, _ = grad(prog, "loss", fin)
print(f"{'None':>5}  {'store all':>13}  {peak_memory(jp0, fin).peak_bytes:>7}  "
      f"{ops_count(jp0, fin).weighted():>7.0f}")
for K in (2, 3, 4, 6, 8, 12, 24):
    jp, _ = grad(prog, "loss", fin, fold_segments=K)
    print(f"{K:>5}  {f'{K} x len {24 // K}':>13}  {peak_memory(jp, fin).peak_bytes:>7}  "
          f"{ops_count(jp, fin).weighted():>7.0f}")

    K       segments   peak B      ops
 None      store all     4168    10325
    2     2 x len 12     2240    11069
    3      3 x len 8     1608    11317
    4      4 x len 6     1384    11441
    6      6 x len 4     1384    11565
    8      8 x len 3     1472    11627
   12     12 x len 2     1736    11689


   24     24 x len 1     2704    11751


The curve: peak falls 4168 → 1384 B (minimum at K = 4–6, √24 ≈ 4.9 — Chen's
heuristic), then *rises* as boundary bookkeeping outweighs the trajectory saved.
Two things matter for what follows. First, **the floor: 1384 B is the best
uniform can do** — no K reaches lower. Second, **the waste**: in the reversed
sweep, once the *last* segment's backward finishes, its boundary state is dead —
and that slot sits **unused** for the rest of the pass. Uniform holds all K
boundaries at once and never reclaims the freed ones.

## The idea: reuse the freed slot

Revolve is what you get by *spending* that freed slot. Reverse a range [lo, hi)
with S checkpoint slots by recursion:

1. Pick a split point c; run the forward step to **store the state at c** (one
   slot).
2. Reverse the **tail** [c, hi) with the remaining **S − 1** slots.
3. The tail is done — c's slot is free again — so reverse the **head** [lo, c)
   with all **S** slots.

A leaf (a single step, or any range that already fits in the slots on hand) is
just the store-all reverse fold from notebook 08. The recursion bottoms out onto
the *same certified piece*; only the ranges and their order are new. Because the
freed slot is handed back to the head, **S slots subdivide a chain far longer
than S** — and how much longer is exactly binomial.

## The binomial rule: $T = \binom{S + r}{S}$

Griewank & Walther's theorem: with **S** snapshots and a budget of **r**
recomputation passes, the longest chain reversible is exactly

$$T_\max(S, r) = \binom{S + r}{S}.$$

It grows fast in both directions — a handful of slots reverses a chain of
hundreds. So for a fixed recompute budget r, the slots needed grow only like
**log T**, where uniform's √T floor needs a slot for every √T steps:

In [4]:
print("        " + "".join(f"r={r:<5}" for r in range(6)))
for S in range(1, 6):
    print(f"S={S:>2}    " + "".join(f"{comb(S + r, S):<7}" for r in range(6)))

        r=0    r=1    r=2    r=3    r=4    r=5    
S= 1    1      2      3      4      5      6      
S= 2    1      3      6      10     15     21     
S= 3    1      4      10     20     35     56     
S= 4    1      5      15     35     70     126    
S= 5    1      6      21     56     126    252    


## The schedule it lays down

`_revolve_split(S, T)` returns where to place the next checkpoint. It is the
optimal offline split — computed here by a small memoized DP over the
recompute-cost recurrence (fold lengths are modest at trace time), landing on
the same optimum revolve's closed form does; ties break toward the longer head
(advance as far as possible). Traced on a short chain (S = 2, T = 8) the shape
is visible: store a checkpoint, recurse the tail with one fewer slot (which
becomes a triangular walk once it is down to a single slot), then reclaim the
slot and recurse the head. The live-checkpoint count never exceeds **S + 1**.

In [5]:
def trace(lo, hi, s, depth, live):
    span, pad = hi - lo, "  " * depth
    if span <= 1 or s >= span:
        print(f"{pad}leaf [{lo},{hi})   (live checkpoints = {live})")
        return
    c = lo + _revolve_split(s, span)
    print(f"{pad}[{lo},{hi}) s={s}: store @{c}, advance {c - lo}; reverse tail then head")
    trace(c, hi, s - 1, depth + 1, live + 1)   # tail: one fewer slot
    trace(lo, c, s, depth + 1, live)           # head: slot reclaimed


trace(0, 8, 2, 0, 1)
print()
print("extra re-advanced steps to reverse T=24 (store-all pays 0):")
for S in (1, 2, 3, 4, 5):
    print(f"  S={S}:  {int(_revolve_cost(S, 24)):>4}   (first split advances "
          f"{_revolve_split(S, 24)} of 24)")

[0,8) s=2: store @5, advance 5; reverse tail then head
  [5,8) s=1: store @7, advance 2; reverse tail then head
    leaf [7,8)   (live checkpoints = 3)
    [5,7) s=1: store @6, advance 1; reverse tail then head
      leaf [6,7)   (live checkpoints = 3)
      leaf [5,6)   (live checkpoints = 2)
  [0,5) s=2: store @2, advance 2; reverse tail then head
    [2,5) s=1: store @4, advance 2; reverse tail then head
      leaf [4,5)   (live checkpoints = 3)
      [2,4) s=1: store @3, advance 1; reverse tail then head
        leaf [3,4)   (live checkpoints = 3)
        leaf [2,3)   (live checkpoints = 2)
    leaf [0,2)   (live checkpoints = 1)

extra re-advanced steps to reverse T=24 (store-all pays 0):
  S=1:   276   (first split advances 23 of 24)
  S=2:    87   (first split advances 18 of 24)
  S=3:    57   (first split advances 15 of 24)
  S=4:    42   (first split advances 12 of 24)
  S=5:    33   (first split advances 12 of 24)


## The three-way comparison

The numbers, on the same FDTD T = 24, `out=final`: store-all against uniform (at
its √T floor) against revolve across a range of slot counts.

In [6]:
prog = fdtd_prog(24)


def peak_ops(**kw):
    jp, _ = grad(prog, "loss", fin, **kw)
    return peak_memory(jp, fin).peak_bytes, ops_count(jp, fin).weighted()


sp, so = peak_ops()
print(f"{'schedule':<20}{'peak B':>8}{'ops':>8}{'vs store':>10}")
print(f"{'store-all':<20}{sp:>8}{so:>8.0f}{'1.00x':>10}")
up = min(peak_ops(fold_segments=K)[0] for K in (4, 6, 8))
uo = min(peak_ops(fold_segments=K)[1] for K in (4, 6, 8))
print(f"{'uniform K~sqrt(24)':<20}{up:>8}{uo:>8.0f}{up / sp:>9.2f}x")
for S in (1, 2, 3, 4, 5):
    p, o = peak_ops(fold_slots=S)
    print(f"{f'revolve S={S}':<20}{p:>8}{o:>8.0f}{p / sp:>9.2f}x")

schedule              peak B     ops  vs store
store-all               4168   10325     1.00x


uniform K~sqrt(24)      1384   11441     0.33x


revolve S=1              856   27437     0.21x


revolve S=2              944   15719     0.23x


revolve S=3             1032   13859     0.25x
revolve S=4             1120   12929     0.27x
revolve S=5             1208   12371     0.29x


Read the **peak** column. Revolve reaches memory points uniform cannot: every
S from 1 to 5 peaks *below* uniform's 1384 B floor, and a **single** slot holds
the peak to 856 B. The peak rises by exactly **88 B per slot** — one (E, H)
state — the O(S · state) live-checkpoint bound made literal. The price is the
**ops** column: S = 1's triangular schedule recomputes heavily (2.66× the
store-all work), and each added slot buys that back down toward the store-all
floor. That is the whole tradeoff dial, and revolve exposes its *optimal*
frontier — √T is merely one point on it, the one uniform is stuck at.

In [7]:
jp0, g0 = grad(prog, "loss", fin)
ref = {v: run(jp0, fin)[g0[v]].to_numpy() for v in ("E0", "H0")}
for S in (1, 2, 3, 4, 5):
    jp, g = grad(prog, "loss", fin, fold_slots=S)
    env = run(jp, fin)
    ok = all(np.allclose(env[g[v]].to_numpy(), ref[v], rtol=1e-9) for v in ("E0", "H0"))
    print(f"S={S}: dL/d(E0,H0) vs store-all -> {'identical (rtol 1e-9)' if ok else 'DIFFERS'}")

S=1: dL/d(E0,H0) vs store-all -> identical (rtol 1e-9)
S=2: dL/d(E0,H0) vs store-all -> identical (rtol 1e-9)


S=3: dL/d(E0,H0) vs store-all -> identical (rtol 1e-9)
S=4: dL/d(E0,H0) vs store-all -> identical (rtol 1e-9)


S=5: dL/d(E0,H0) vs store-all -> identical (rtol 1e-9)


## The no-divisibility win

`fold_segments=K` needs K to **divide** T — a uniform stride is meaningless
otherwise, and notebook 11 refused an indivisible K loudly. Revolve carries no
such constraint: the recursive split handles **any** T. Take T = 13, prime, so
*no* interior K divides it:

In [8]:
prog13 = fdtd_prog(13)
try:
    grad(prog13, "loss", fin, fold_segments=4)
except ValueError as e:
    print("fold_segments=4 :", e)

jp0, g0 = grad(prog13, "loss", fin)
ref = {v: run(jp0, fin)[g0[v]].to_numpy() for v in ("E0", "H0")}
jp, g = grad(prog13, "loss", fin, fold_slots=3)
env = run(jp, fin)
ok = all(np.allclose(env[g[v]].to_numpy(), ref[v], rtol=1e-9) for v in ("E0", "H0"))
print(f"fold_slots=3    : reversed T=13, gradient "
      f"{'identical' if ok else 'DIFFERS'} to store-all")
print(f"                 peak {peak_memory(jp0, fin).peak_bytes} B (store-all) "
      f"-> {peak_memory(jp, fin).peak_bytes} B (revolve S=3)")

fold_segments=4 : fold_segments=4 must divide the fold extent 13 (pad the dim or pick a divisor)
fold_slots=3    : reversed T=13, gradient identical to store-all
                 peak 2320 B (store-all) -> 1032 B (revolve S=3)


## With per-step elements, too

FDTD is extent-driven (no per-step inputs). Revolve reuses the *same* seam
machinery for a fold **with** elements — gated linear attention's matrix state
$S_t = a_t S_{t-1} + k_t v_t^\top$, five inputs, an emitted trajectory. The
element cotangents are sliced per range, padded back to full extent, and
accumulated through `contribute`'s fan-in — exactly as the uniform path does.

In [9]:
DK, DV, TN = 3, 2, 4
RNG = np.random.default_rng(23)
GLA_STEP = Program(
    (
        I("S", "input"), I("a", "input"), I("kk", "input"), I("vv", "input"), I("qq", "input"),
        I("a1", "repeat", ["a"], name="p", extent=(0, DK)),
        I("ar", "repeat", ["a1"], name="r", extent=(0, DV)),
        I("kr", "repeat", ["kk"], name="r", extent=(0, DV)),
        I("vr", "repeat", ["vv"], name="p", extent=(0, DK)),
        I("Sa", "pointwise", ["ar", "S"], f="mul"),
        I("kv", "pointwise", ["kr", "vr"], f="mul"),
        I("S1", "pointwise", ["Sa", "kv"], f="add"),
        I("qr", "repeat", ["qq"], name="r", extent=(0, DV)),
        I("Sq", "pointwise", ["S1", "qr"], f="mul"),
        I("y", "reduce", ["Sq"], f="sum", dims=("p",)),
    )
)
gla = Program(
    (
        I("S0", "input"), I("a", "input"), I("k", "input"), I("v", "input"), I("q", "input"),
        I("ys", "fold", ["S0", "a", "k", "v", "q"], step=GLA_STEP, dim="t",
          state=("S",), element=("a", "kk", "vv", "qq"), carry={"S": "S1"}, out=("emit", "y")),
        I("y2", "pointwise", ["ys", "ys"], f="mul"),
        I("loss", "reduce", ["y2"], f="sum", dims=("t", "r")),
    )
)
gin = {
    "S0": T(RNG.standard_normal((DK, DV)), ("p", "r")),
    "a": T(RNG.uniform(0.5, 1.0, TN), ("t",)),
    "k": T(RNG.standard_normal((TN, DK)), ("t", "p")),
    "v": T(RNG.standard_normal((TN, DV)), ("t", "r")),
    "q": T(RNG.standard_normal((TN, DK)), ("t", "p")),
}
jp0, g0 = grad(gla, "loss", gin)
e0 = run(jp0, gin)
for S in (1, 2, 3):
    jp, g = grad(gla, "loss", gin, fold_slots=S)
    e = run(jp, gin)
    ok = all(np.allclose(e[g[v]].to_numpy(order=gin[v].names),
                         e0[g0[v]].to_numpy(order=gin[v].names), rtol=1e-9)
             for v in ("S0", "a", "k", "v", "q"))
    print(f"GLA fold_slots={S}: dL/d(S0,a,k,v,q) -> {'all identical' if ok else 'DIFFERS'}")

GLA fold_slots=1: dL/d(S0,a,k,v,q) -> all identical
GLA fold_slots=2: dL/d(S0,a,k,v,q) -> all identical


GLA fold_slots=3: dL/d(S0,a,k,v,q) -> all identical


## What revolve closes — and what stays honest

Everything CONCERNS.md #27 flagged as open for the *binomial* piece now lands:
revolve is optimal (the full $\binom{S+r}{S}$ frontier), lower-than-√T memory is
reachable, and there is no divisibility constraint on T. Two caveats stay
accurate and unchanged: the slot count S is **global per `grad` call**, not yet
per-fold; and the peak/recompute figures are the reference layer's model
(uniform 8-byte items; the fold transient an upper bound) — coarse, but
identical across every row of the comparison, so the *ratios* are honest.

The deeper point is the one the LEAN diary makes: **revolve is a *schedule* over
the same lemmas.** Correctness lives entirely in the certified backward step —
the reverse fold, the s_{t−1} reconstruction, the ŝ seam identity — proved once
in notebook 08 and reused verbatim. `fold_slots` only reorders and re-times the
ranges those lemmas run over. Strategy and correctness factor cleanly: the DP
that picks the splits could be *wrong* and the gradients would still be exact
(just slower, or fatter), because the schedule cannot touch what the pieces
compute. That separation is why one `grad` returns bit-identical gradients
across store-all, uniform, and revolve — the whole comparison was
apples-to-apples from the first row.